# Diagnóstico de cancro da mama — Active Learning com Human-in-the-Loop

**Inteligência Artificial Centrada no Humano** · Mestrado em Inteligência Artificial e Engenharia de Dados · FCTUC

Começa com poucas imagens anotadas e, em cada iteração, o modelo escolhe a imagem mais útil e pede a label ao humano. Compara três estratégias de query do modAL (`uncertainty`, `entropy`, `margin`) com vários batch sizes. Correr primeiro `01_passive_learning.ipynb` para ter a baseline.

**Índice**
1. Configuração
2. Dados
3. Experiências
4. Resultados
5. Conclusões

## 1. Configuração

In [ ]:
# Ambiente: monta o Google Drive no Colab; localmente usa a pasta do repositório
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
from src.config import DATA_PATH, RESULTS_PATH

In [ ]:
# Parâmetros
IMG_SIZE = 224          # reduzir (ex.: 128) se faltar memória
MAX_PER_CLASS = None    # limitar o nº de imagens originais por classe (None = todas)
STRATEGIES = ["uncertainty", "entropy", "margin"]
BATCH_SIZES = [16, 32, 64, 128]
N_INITIAL = 1000        # nº de imagens anotadas no início
N_QUERIES = 10          # nº de iterações de Active Learning (imagens pedidas ao humano)
EPOCHS = 10
HUMAN = False           # True: pede a label a uma pessoa; False: simula o humano com a label do dataset

## 2. Dados

In [ ]:
# Carregar, dividir em treino/teste e balancear o treino (o teste fica com as imagens originais)
import numpy as np
from src.data import CLASSES, load_dataset, summarize

x_train, x_test, y_train, y_test = load_dataset(DATA_PATH, size=IMG_SIZE, max_per_class=MAX_PER_CLASS)
summarize(x_train, x_test, y_train, y_test)

## 3. Experiências

O ciclo de Active Learning está em `src/active_learning.py`. Com `HUMAN = True`, a imagem escolhida é mostrada com a sugestão do modelo e a pessoa escreve a classe correta; o modelo aprende com essa resposta. Os resultados ficam em `results/active_learning.json`.

In [ ]:
from src.active_learning import run_active_learning
from src.evaluation import compute_metrics, save_json

results = {}   # results[estratégia][batch_size] = {"history", "y_pred", "metrics"}
for strategy in STRATEGIES:
    results[strategy] = {}
    for batch_size in BATCH_SIZES:
        print(f"{strategy} - batch {batch_size}")
        history, y_pred = run_active_learning(strategy, x_train, y_train, x_test, y_test,
                                              batch_size=batch_size, epochs=EPOCHS, n_initial=N_INITIAL,
                                              n_queries=N_QUERIES, human=HUMAN)
        results[strategy][batch_size] = {"history": history, "y_pred": y_pred.tolist(),
                                         "metrics": compute_metrics(y_test, y_pred)}

save_json(results, RESULTS_PATH / "active_learning.json")

## 4. Resultados

### 4.1 Evolução ao longo das iterações

A linha preta tracejada é a baseline de Passive Learning.

In [ ]:
from src.evaluation import load_json, plot_history

baseline_file = RESULTS_PATH / "passive_learning.json"
baseline = load_json(baseline_file) if baseline_file.exists() else None

for strategy in STRATEGIES:
    histories = {batch: r["history"] for batch, r in results[strategy].items()}
    plot_history(histories, f"Active Learning using {strategy}_sampling", baseline=baseline)

### 4.2 Métricas no conjunto de teste

FN = maligno classificado como benigno ou normal.

In [ ]:
print(f"{'estratégia':<12}{'batch':>6}{'accuracy':>10}{'recall':>8}{'F1':>8}{'recall mal.':>13}{'FN':>5}")
rows = [("passive", "-", baseline)] if baseline is not None else []
rows += [(s, b, r["metrics"]) for s in STRATEGIES for b, r in results[s].items()]
for strategy, batch_size, m in rows:
    print(f"{strategy:<12}{batch_size:>6}{m['accuracy']:>10.3f}{m['recall_macro']:>8.3f}{m['f1_macro']:>8.3f}"
          f"{m['recall_malignant']:>13.3f}{m['false_negatives']:>5}")

### 4.3 Matrizes de confusão da estratégia com menos falsos negativos

In [ ]:
from src.evaluation import plot_confusion_matrix

best = min(STRATEGIES, key=lambda s: min(r["metrics"]["false_negatives"] for r in results[s].values()))
for batch_size, r in results[best].items():
    plot_confusion_matrix(y_test, r["y_pred"], title=f"{best}_sampling - batch {batch_size}")

## 5. Conclusões

*A preencher depois de correr as experiências completas.*